<a href="https://colab.research.google.com/github/vij5566/Churn-Prediction-Project/blob/main/StructuredDataExtraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install unsloth

In [9]:
# train.py
import torch
import json
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig
#from transformers import TrainingArguments
import wandb

In [4]:
# 1. Initialize Weights & Biases for tracking
wandb.init(project="llama3-invoice-extraction")

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: n210710 (n210710-rajiv-gandhi-university-of-knowledge-technologies) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai


In [5]:
# 2. Load the base model in 4-bit quantization using Unsloth
max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

==((====))==  Unsloth 2026.7.1: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3-8b-bnb-4bit as a legacy tokenizer.


In [6]:
# 3. Apply LoRA Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Your chosen rank
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 32, # Your chosen alpha
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
)

Unsloth 2026.7.1 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [7]:
# 4. Pull and Format the CORD Dataset from Hugging Face
prompt_template = """Instruction: Extract the store name, menu items, and total price from this receipt text. Return a JSON object.

Input Text: {}

Response: {}"""

def format_cord_prompts(examples):
    texts = []
    # Iterate through the batch of examples from CORD
    for ground_truth_str in examples["ground_truth"]:
        # CORD stores its ground truth as a JSON string containing the parsed data
        data = json.loads(ground_truth_str)

        # 4a. Reconstruct the messy OCR text (Input)
        valid_lines = data.get("valid_line", [])
        raw_text_parts = []
        for line in valid_lines:
            for word in line.get("words", []):
                raw_text_parts.append(word.get("text", ""))
        raw_text = " ".join(raw_text_parts)

       # 4b. Extract the clean labels (Response) - SAFELY
        gt_dict = data.get("gt_parse")
        if not isinstance(gt_dict, dict):
            gt_dict = {}

        # 1. Safely handle store info
        store_info = gt_dict.get("store_info")
        if isinstance(store_info, dict):
            store_name = store_info.get("name", "")
        else:
            store_name = ""

        # 2. Safely handle menu items
        menu_info = gt_dict.get("menu")
        menu_items = []
        if isinstance(menu_info, list):
            for item in menu_info:
                # Only use .get() if we are absolutely sure it is a dictionary
                if isinstance(item, dict) and item.get("nm"):
                    menu_items.append(item.get("nm"))
                # If the dataset accidentally just gave us a string, save the string directly
                elif isinstance(item, str):
                    menu_items.append(item)

        # 3. Safely handle totals
        total_info = gt_dict.get("total")
        if isinstance(total_info, dict):
            total_price = total_info.get("total_price", "")
        else:
            total_price = ""

        # Build the final structured JSON response
        target_json = json.dumps({
            "store_name": store_name,
            "menu_items": menu_items,
            "total_price": total_price
        })

        # 4c. Combine into the Prompt-Response template
        text = prompt_template.format(raw_text, target_json) + tokenizer.eos_token
        texts.append(text)

    return { "text" : texts }


print("Downloading and formatting dataset...")
dataset = load_dataset("naver-clova-ix/cord-v2", split="train")

# Added remove_columns to drop images and prevent vision-model errors
dataset = dataset.map(
    format_cord_prompts,
    batched = True,
    remove_columns = dataset.column_names
)

In [10]:
# 5. Set up the SFTTrainer
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    args = SFTConfig(   # Changed from TrainingArguments to SFTConfig
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "wandb",
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/800 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [11]:
# 6. Train and Save
trainer.train()
model.save_pretrained("lora_model") # Saves your fine-tuned adapters
tokenizer.save_pretrained("lora_model")

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 800 | Num Epochs = 3 | Total steps = 300
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,2.391211
2,2.351602
3,2.514034
4,2.438146
5,2.173063
6,2.249046
7,2.110689
8,1.603585
9,1.826192
10,1.727955


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-300/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in lora_model/tokenizer_config.json.


('lora_model/tokenizer_config.json', 'lora_model/tokenizer.json')